In [1]:
from google.colab import drive
drive.mount('/content/drive')

!pip install librosa soundfile -q

import os
import json
import pickle
import numpy as np
import pandas as pd
from pathlib import Path
from tqdm.auto import tqdm
import librosa
from sklearn.preprocessing import StandardScaler

PROJECT_ROOT = Path('/content/drive/MyDrive/TA_SER')


CSV_PATH          = PROJECT_ROOT / 'data' / 'dataset_split_v4.csv'
FEATURES_ROOT     = PROJECT_ROOT / 'data' / 'processed' / 'features_v4'
SCALER_DIR        = PROJECT_ROOT / 'data' / 'scalers'
LABEL_ENCODER_FP  = PROJECT_ROOT / 'data' / 'processed' / 'label_encoder.json'

SR             = 16000
DURATION_SEC   = 3.0
TARGET_SAMPLES = int(SR * DURATION_SEC)
TOP_DB         = 20

N_MFCC     = 40
N_FFT      = 512
HOP_LENGTH = 256
EXPECTED_TIME_FRAMES = 188

NOISE_RATIO          = 0.005
TIME_STRETCH_RANGE   = (0.85, 1.15)
PITCH_SHIFT_RANGE    = (-3, 3)

TARGET_LABELS = ['angry', 'happy', 'neutral', 'sad']
LABEL_TO_INT  = {lbl: idx for idx, lbl in enumerate(TARGET_LABELS)}
INT_TO_LABEL  = {idx: lbl for lbl, idx in LABEL_TO_INT.items()}

SEED = 42
np.random.seed(SEED)

(FEATURES_ROOT / 'train').mkdir(parents=True, exist_ok=True)
(FEATURES_ROOT / 'val').mkdir(parents=True, exist_ok=True)
(FEATURES_ROOT / 'test').mkdir(parents=True, exist_ok=True)
SCALER_DIR.mkdir(parents=True, exist_ok=True)

print(f'CSV path       : {CSV_PATH}')
print(f'Features output: {FEATURES_ROOT}')
print(f'Scaler output  : {SCALER_DIR}')
print(f'MFCC shape exp : ({EXPECTED_TIME_FRAMES}, {N_MFCC})')
print(f'Label encoding : {LABEL_TO_INT}')


Mounted at /content/drive
CSV path       : /content/drive/MyDrive/TA_SER/data/dataset_split_v4.csv
Features output: /content/drive/MyDrive/TA_SER/data/processed/features_v4
Scaler output  : /content/drive/MyDrive/TA_SER/data/scalers
MFCC shape exp : (188, 40)
Label encoding : {'angry': 0, 'happy': 1, 'neutral': 2, 'sad': 3}


In [2]:
with open(LABEL_ENCODER_FP, 'w') as f:
    json.dump({
        'label_to_int': LABEL_TO_INT,
        'int_to_label': {str(k): v for k, v in INT_TO_LABEL.items()},
        'target_labels': TARGET_LABELS
    }, f, indent=2)

print(f'Label encoder saved: {LABEL_ENCODER_FP}')
print(json.dumps({'label_to_int': LABEL_TO_INT}, indent=2))


Label encoder saved: /content/drive/MyDrive/TA_SER/data/processed/label_encoder.json
{
  "label_to_int": {
    "angry": 0,
    "happy": 1,
    "neutral": 2,
    "sad": 3
  }
}


In [4]:
def load_and_preprocess_audio(filepath, sr=SR, target_samples=TARGET_SAMPLES, top_db=TOP_DB):

    y, _ = librosa.load(filepath, sr=sr, mono=True)

    y, _ = librosa.effects.trim(y, top_db=top_db)

    max_abs = np.max(np.abs(y))
    if max_abs > 0:
        y = y / max_abs

    if len(y) < target_samples:
        pad_total = target_samples - len(y)
        pad_left  = pad_total // 2
        pad_right = pad_total - pad_left
        y = np.pad(y, (pad_left, pad_right), mode='constant')

    elif len(y) > target_samples:
        start = (len(y) - target_samples) // 2
        y = y[start:start + target_samples]

    return y.astype(np.float32)


sample_row = pd.read_csv(CSV_PATH).iloc[0]
test_audio = load_and_preprocess_audio(sample_row['filepath'])
print(f'Test audio: len={len(test_audio)}  min={test_audio.min():.3f}  max={test_audio.max():.3f}  dtype={test_audio.dtype}')
assert len(test_audio) == TARGET_SAMPLES, f'Expected {TARGET_SAMPLES}, got {len(test_audio)}'
print(' Audio preprocessing OK')


Test audio: len=48000  min=-1.000  max=0.971  dtype=float32
 Audio preprocessing OK


In [ ]:
def extract_mfcc(audio, sr=SR, n_mfcc=N_MFCC, n_fft=N_FFT, hop_length=HOP_LENGTH):

    mfcc = librosa.feature.mfcc(
        y=audio,
        sr=sr,
        n_mfcc=n_mfcc,
        n_fft=n_fft,
        hop_length=hop_length
    )
    return mfcc.T.astype(np.float32)


test_mfcc = extract_mfcc(test_audio)
print(f'Test MFCC shape: {test_mfcc.shape}  dtype: {test_mfcc.dtype}')
print(f'  Expected: ({EXPECTED_TIME_FRAMES}, {N_MFCC})')
print(f'  Mean: {test_mfcc.mean():.3f}  Std: {test_mfcc.std():.3f}')
assert test_mfcc.shape == (EXPECTED_TIME_FRAMES, N_MFCC), \
    f'Shape mismatch: got {test_mfcc.shape}, expected ({EXPECTED_TIME_FRAMES}, {N_MFCC})'
print(' MFCC extraction OK')


Test MFCC shape: (188, 40)  dtype: float32
  Expected: (188, 40)
  Mean: -13.032  Std: 76.649
 MFCC extraction OK


In [ ]:
def aug_noise(audio, ratio=NOISE_RATIO):

    max_val = np.max(np.abs(audio))
    noise_factor = ratio * max_val * np.random.uniform()
    noise = np.random.randn(len(audio)).astype(np.float32)
    return (audio + noise_factor * noise).astype(np.float32)


def aug_time_stretch(audio, rate_range=TIME_STRETCH_RANGE, target_samples=TARGET_SAMPLES):

    rate = np.random.uniform(*rate_range)
    stretched = librosa.effects.time_stretch(audio, rate=rate)

    if len(stretched) < target_samples:
        pad_total = target_samples - len(stretched)
        pad_left  = pad_total // 2
        pad_right = pad_total - pad_left
        stretched = np.pad(stretched, (pad_left, pad_right), mode='constant')
    elif len(stretched) > target_samples:
        start = (len(stretched) - target_samples) // 2
        stretched = stretched[start:start + target_samples]
    return stretched.astype(np.float32)


def aug_pitch_shift(audio, sr=SR, n_steps_range=PITCH_SHIFT_RANGE):

    n_steps = np.random.uniform(*n_steps_range)
    shifted = librosa.effects.pitch_shift(audio, sr=sr, n_steps=n_steps)
    return shifted.astype(np.float32)


print('Testing augmentations on 1 sample')
for name, fn in [('noise', aug_noise), ('stretch', aug_time_stretch), ('pitch', aug_pitch_shift)]:
    aug_audio = fn(test_audio)
    assert len(aug_audio) == TARGET_SAMPLES, f'{name}: length mismatch {len(aug_audio)}'
    aug_mfcc = extract_mfcc(aug_audio)
    assert aug_mfcc.shape == (EXPECTED_TIME_FRAMES, N_MFCC)
    print(f' {name:8s}: audio_len={len(aug_audio)}, mfcc_shape={aug_mfcc.shape}')
print(' All augmentations OK')


Testing augmentations on 1 sample
 noise   : audio_len=48000, mfcc_shape=(188, 40)
 stretch : audio_len=48000, mfcc_shape=(188, 40)
 pitch   : audio_len=48000, mfcc_shape=(188, 40)
 All augmentations OK


In [ ]:
def extract_features_batch(df, augment=False, desc='Extracting'):

    X_list, y_list, sources_list = [], [], []

    for _, row in tqdm(df.iterrows(), total=len(df), desc=desc):
        try:
            audio = load_and_preprocess_audio(row['filepath'])
        except Exception as e:
            print(f"\n Failed to load {row['filepath']}: {e}")
            continue

        label_int = LABEL_TO_INT[row['label']]
        source    = row['source']

        X_list.append(extract_mfcc(audio))
        y_list.append(label_int)
        sources_list.append(source)

        if augment:

            X_list.append(extract_mfcc(aug_noise(audio)))
            y_list.append(label_int)
            sources_list.append(source)


            X_list.append(extract_mfcc(aug_time_stretch(audio)))
            y_list.append(label_int)
            sources_list.append(source)


            X_list.append(extract_mfcc(aug_pitch_shift(audio)))
            y_list.append(label_int)
            sources_list.append(source)

    X = np.array(X_list, dtype=np.float32)
    y = np.array(y_list, dtype=np.int32)
    sources = np.array(sources_list, dtype=object)

    return X, y, sources


In [ ]:
df_all = pd.read_csv(CSV_PATH)
print(f'Loaded CSV: {df_all.shape}')

assert set(df_all['label'].unique()) == set(TARGET_LABELS), 'Label mismatch!'
assert set(df_all['split'].unique()) == {'train', 'val', 'test'}, 'Split mismatch!'

df_train = df_all[df_all['split'] == 'train'].reset_index(drop=True)
df_val   = df_all[df_all['split'] == 'val'].reset_index(drop=True)
df_test  = df_all[df_all['split'] == 'test'].reset_index(drop=True)

print(f'\\nTrain: {len(df_train)} files  → akan jadi {len(df_train) * 4} samples setelah aug 4x')
print(f'Val  : {len(df_val)} files  (no aug)')
print(f'Test : {len(df_test)} files  (no aug)')

print(f'\\n--- Train distribution ---')
print(df_train.groupby(['source', 'label']).size().unstack(fill_value=0))


Loaded CSV: (2911, 4)
\nTrain: 2328 files  → akan jadi 9312 samples setelah aug 4x
Val  : 291 files  (no aug)
Test : 292 files  (no aug)
\n--- Train distribution ---
label    angry  happy  neutral  sad
source                             
emodb      101     57       63   50
ravdess    154    153       77  153
savee       48     48       96   48
tess       320    320      320  320


In [ ]:
X_train, y_train, sources_train = extract_features_batch(
    df_train, augment=True, desc='Train (with aug)'
)

print(f'\\n--- Train shapes ---')
print(f'X_train : {X_train.shape}  dtype={X_train.dtype}')
print(f'y_train : {y_train.shape}  unique={np.unique(y_train)}')
print(f'sources : {sources_train.shape}  unique={np.unique(sources_train)}')

# Verify distribusi
print(f'\n Distribusi kelas di train (setelah aug)')
unique, counts = np.unique(y_train, return_counts=True)
for u, c in zip(unique, counts):
    print(f'  {INT_TO_LABEL[u]:8s} ({u}): {c}')

print(f'\n Distribusi source di train (setelah aug)')
unique, counts = np.unique(sources_train, return_counts=True)
for u, c in zip(unique, counts):
    print(f'  {u:8s}: {c}')


Train (with aug):   0%|          | 0/2328 [00:00<?, ?it/s]

\n--- Train shapes ---
X_train : (9312, 188, 40)  dtype=float32
y_train : (9312,)  unique=[0 1 2 3]
sources : (9312,)  unique=['emodb' 'ravdess' 'savee' 'tess']

 Distribusi kelas di train (setelah aug)
  angry    (0): 2492
  happy    (1): 2312
  neutral  (2): 2224
  sad      (3): 2284

 Distribusi source di train (setelah aug)
  emodb   : 1084
  ravdess : 2148
  savee   : 960
  tess    : 5120


In [ ]:
X_val, y_val, sources_val = extract_features_batch(
    df_val, augment=False, desc='Val (no aug)'
)
print(f'\n X_val: {X_val.shape}  y_val: {y_val.shape}  sources_val: {sources_val.shape}')


Val (no aug):   0%|          | 0/291 [00:00<?, ?it/s]


 X_val: (291, 188, 40)  y_val: (291,)  sources_val: (291,)


In [ ]:
X_test, y_test, sources_test = extract_features_batch(
    df_test, augment=False, desc='Test (no aug)'
)
print(f'\n X_test: {X_test.shape}  y_test: {y_test.shape}  sources_test: {sources_test.shape}')


Test (no aug):   0%|          | 0/292 [00:00<?, ?it/s]


 X_test: (292, 188, 40)  y_test: (292,)  sources_test: (292,)


In [ ]:
scalers = {}

print('Fitting & applying per-dataset StandardScaler \n')

X_train_scaled = np.zeros_like(X_train)
X_val_scaled   = np.zeros_like(X_val)
X_test_scaled  = np.zeros_like(X_test)

unique_sources = sorted(np.unique(sources_train))

for src in unique_sources:

    mask_tr = (sources_train == src)
    mask_vl = (sources_val == src)
    mask_ts = (sources_test == src)

    n_tr = int(mask_tr.sum())
    n_vl = int(mask_vl.sum())
    n_ts = int(mask_ts.sum())

    if n_tr == 0:
        print(f' Skip {src}: no train samples')
        continue

    scaler = StandardScaler()

    X_tr_src  = X_train[mask_tr]
    X_tr_flat = X_tr_src.reshape(-1, N_MFCC)
    scaler.fit(X_tr_flat)

    X_train_scaled[mask_tr] = scaler.transform(X_tr_flat).reshape(X_tr_src.shape)

    if n_vl > 0:
        X_vl_src  = X_val[mask_vl]
        X_vl_flat = X_vl_src.reshape(-1, N_MFCC)
        X_val_scaled[mask_vl] = scaler.transform(X_vl_flat).reshape(X_vl_src.shape)

    if n_ts > 0:
        X_ts_src  = X_test[mask_ts]
        X_ts_flat = X_ts_src.reshape(-1, N_MFCC)
        X_test_scaled[mask_ts] = scaler.transform(X_ts_flat).reshape(X_ts_src.shape)

    scalers[src] = scaler
    print(f'  {src:8s}: train={n_tr:5d}  val={n_vl:3d}  test={n_ts:3d}'
          f'mean[0:3]={scaler.mean_[:3].round(2)}')

print(f'\n {len(scalers)} scalers fitted')

print(f'\n Train scaled stats per source:')
for src in scalers.keys():
    mask = (sources_train == src)
    X_s = X_train_scaled[mask].reshape(-1, N_MFCC)
    print(f'  {src:8s}: mean={X_s.mean():+.4f} (≈0)  std={X_s.std():.4f} (≈1)')


Fitting & applying per-dataset StandardScaler 

  emodb   : train= 1084  val= 34  test= 34mean[0:3]=[-428.12   51.16   -2.94]
  ravdess : train= 2148  val= 67  test= 68mean[0:3]=[-469.78   45.08   -8.65]
  savee   : train=  960  val= 30  test= 30mean[0:3]=[-430.24  109.91   27.11]
  tess    : train= 5120  val=160  test=160mean[0:3]=[-470.75   49.55   10.01]

 4 scalers fitted

 Train scaled stats per source:
  emodb   : mean=+0.0000 (≈0)  std=1.0000 (≈1)
  ravdess : mean=+0.0000 (≈0)  std=1.0000 (≈1)
  savee   : mean=+0.0000 (≈0)  std=1.0000 (≈1)
  tess    : mean=-0.0000 (≈0)  std=1.0000 (≈1)


In [ ]:
rng = np.random.default_rng(SEED)
shuffle_idx = rng.permutation(len(X_train_scaled))

X_train_final       = X_train_scaled[shuffle_idx]
y_train_final       = y_train[shuffle_idx]
sources_train_final = sources_train[shuffle_idx]

train_dir = FEATURES_ROOT / 'train'
np.save(train_dir / 'X_train.npy', X_train_final)
np.save(train_dir / 'y_train.npy', y_train_final)
np.save(train_dir / 'sources_train.npy', sources_train_final)

print(f' Saved to {train_dir}/')
print(f' X_train.npy      : {X_train_final.shape}  ({X_train_final.nbytes / 1e6:.1f} MB)')
print(f' y_train.npy      : {y_train_final.shape}')
print(f' sources_train.npy: {sources_train_final.shape}')


 Saved to /content/drive/MyDrive/TA_SER/data/processed/features_v4/train/
 X_train.npy      : (9312, 188, 40)  (280.1 MB)
 y_train.npy      : (9312,)
 sources_train.npy: (9312,)


In [ ]:
val_dir = FEATURES_ROOT / 'val'

print('Val per-source:')
for src in unique_sources:
    mask = (sources_val == src)
    np.save(val_dir / f'X_val_{src}.npy', X_val_scaled[mask])
    np.save(val_dir / f'y_val_{src}.npy', y_val[mask])
    print(f'  {src:8s}: X={X_val_scaled[mask].shape}  y={y_val[mask].shape}')

combined_idx = rng.permutation(len(X_val_scaled))
np.save(val_dir / 'X_val_combined.npy',       X_val_scaled[combined_idx])
np.save(val_dir / 'y_val_combined.npy',       y_val[combined_idx])
np.save(val_dir / 'sources_val_combined.npy', sources_val[combined_idx])
print(f'\n combined: X={X_val_scaled.shape}  y={y_val.shape}  sources={sources_val.shape}')

print(f'\n Saved to {val_dir}/')


Val per-source:
  emodb   : X=(34, 188, 40)  y=(34,)
  ravdess : X=(67, 188, 40)  y=(67,)
  savee   : X=(30, 188, 40)  y=(30,)
  tess    : X=(160, 188, 40)  y=(160,)

 combined: X=(291, 188, 40)  y=(291,)  sources=(291,)

 Saved to /content/drive/MyDrive/TA_SER/data/processed/features_v4/val/


In [ ]:
test_dir = FEATURES_ROOT / 'test'

print('Test per-source:')
for src in unique_sources:
    mask = (sources_test == src)
    np.save(test_dir / f'X_test_{src}.npy', X_test_scaled[mask])
    np.save(test_dir / f'y_test_{src}.npy', y_test[mask])
    print(f'  {src:8s}: X={X_test_scaled[mask].shape}  y={y_test[mask].shape}')

combined_idx = rng.permutation(len(X_test_scaled))
np.save(test_dir / 'X_test_combined.npy',       X_test_scaled[combined_idx])
np.save(test_dir / 'y_test_combined.npy',       y_test[combined_idx])
np.save(test_dir / 'sources_test_combined.npy', sources_test[combined_idx])
print(f'\n combined: X={X_test_scaled.shape}  y={y_test.shape}  sources={sources_test.shape}')

print(f'\n Saved to {test_dir}/')


Test per-source:
  emodb   : X=(34, 188, 40)  y=(34,)
  ravdess : X=(68, 188, 40)  y=(68,)
  savee   : X=(30, 188, 40)  y=(30,)
  tess    : X=(160, 188, 40)  y=(160,)

 combined: X=(292, 188, 40)  y=(292,)  sources=(292,)

 Saved to /content/drive/MyDrive/TA_SER/data/processed/features_v4/test/


In [ ]:
for src, scaler in scalers.items():
    fp = SCALER_DIR / f'scaler_{src}.pkl'
    with open(fp, 'wb') as f:
        pickle.dump(scaler, f)
    print(f' {fp.name}')

print(f'\n {len(scalers)} scalers saved to {SCALER_DIR}/')


 scaler_emodb.pkl
 scaler_ravdess.pkl
 scaler_savee.pkl
 scaler_tess.pkl

 4 scalers saved to /content/drive/MyDrive/TA_SER/data/scalers/


In [ ]:
print('=' * 60)
print('VERIFY TRAIN')
print('=' * 60)
X_tr = np.load(FEATURES_ROOT / 'train' / 'X_train.npy')
y_tr = np.load(FEATURES_ROOT / 'train' / 'y_train.npy')
s_tr = np.load(FEATURES_ROOT / 'train' / 'sources_train.npy', allow_pickle=True)
print(f'X_train: {X_tr.shape}  dtype={X_tr.dtype}  range=[{X_tr.min():.2f}, {X_tr.max():.2f}]')
print(f'y_train: {y_tr.shape}  unique={np.unique(y_tr, return_counts=True)}')
print(f'sources: {s_tr.shape}  unique={np.unique(s_tr, return_counts=True)}')
assert X_tr.shape[1:] == (EXPECTED_TIME_FRAMES, N_MFCC)
assert X_tr.dtype == np.float32

print('\\n' + '=' * 60)
print('VERIFY VAL per-source')
print('=' * 60)
for src in unique_sources:
    X = np.load(FEATURES_ROOT / 'val' / f'X_val_{src}.npy')
    y = np.load(FEATURES_ROOT / 'val' / f'y_val_{src}.npy')
    print(f'  {src:8s}: X={X.shape}  y={y.shape}  y_dist={dict(zip(*np.unique(y, return_counts=True)))}')

print('\\n' + '=' * 60)
print('VERIFY TEST per-source')
print('=' * 60)
for src in unique_sources:
    X = np.load(FEATURES_ROOT / 'test' / f'X_test_{src}.npy')
    y = np.load(FEATURES_ROOT / 'test' / f'y_test_{src}.npy')
    print(f'  {src:8s}: X={X.shape}  y={y.shape}  y_dist={dict(zip(*np.unique(y, return_counts=True)))}')

print('\\n' + '=' * 60)
print('VERIFY SCALERS')
print('=' * 60)
for src in unique_sources:
    fp = SCALER_DIR / f'scaler_{src}.pkl'
    with open(fp, 'rb') as f:
        sc = pickle.load(f)
    print(f'  {src:8s}: mean_shape={sc.mean_.shape}  mean[0:3]={sc.mean_[:3].round(2)}')

print('\n ALL VERIFIED')


VERIFY TRAIN
X_train: (9312, 188, 40)  dtype=float32  range=[-6.62, 9.88]
y_train: (9312,)  unique=(array([0, 1, 2, 3], dtype=int32), array([2492, 2312, 2224, 2284]))
sources: (9312,)  unique=(array(['emodb', 'ravdess', 'savee', 'tess'], dtype=object), array([1084, 2148,  960, 5120]))
\n============================================================
VERIFY VAL per-source
  emodb   : X=(34, 188, 40)  y=(34,)  y_dist={np.int32(0): np.int64(13), np.int32(1): np.int64(7), np.int32(2): np.int64(8), np.int32(3): np.int64(6)}
  ravdess : X=(67, 188, 40)  y=(67,)  y_dist={np.int32(0): np.int64(19), np.int32(1): np.int64(19), np.int32(2): np.int64(10), np.int32(3): np.int64(19)}
  savee   : X=(30, 188, 40)  y=(30,)  y_dist={np.int32(0): np.int64(6), np.int32(1): np.int64(6), np.int32(2): np.int64(12), np.int32(3): np.int64(6)}
  tess    : X=(160, 188, 40)  y=(160,)  y_dist={np.int32(0): np.int64(40), np.int32(1): np.int64(40), np.int32(2): np.int64(40), np.int32(3): np.int64(40)}
\n===============